# Przegląd modelu jakości (zdjęcia bez oceny EyeQ)

Model: regresja logistyczna na cechach DenseNet121 (ekstraktor z QuickQual), uczona na ręcznych
ocenach EyeQ (Reject vs reszta). Wyniki z `scripts/fit_quality_model.py` →
`<EYEPACS_QUALITY>/quality_scores.csv`.

`p_reject` = prawdopodobieństwo, że EyeQ oceniłby zdjęcie jako Reject. Dla zdjęć z oceną EyeQ to
predykcja z walidacji krzyżowej (model ich nie widział), dla pozostałych — z modelu uczonego na wszystkich ocenionych.

**Cel:** obejrzeć, co model odrzuca wśród ~22,6k zdjęć bez ręcznej oceny, zanim użyjemy go w manifeście v2.

In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CACHE_DIR = Path(os.environ.get('EYEPACS_CACHE', '~/data/eyepacs/cache_768')).expanduser()
QUALITY_DIR = Path(os.environ.get('EYEPACS_QUALITY', '~/data/eyepacs/quality')).expanduser()
IMAGES_DIR = CACHE_DIR / 'images'

THRESHOLD = 0.5
SEED = 0

scores = pd.read_csv(QUALITY_DIR / 'quality_scores.csv', keep_default_na=False)
manifest = pd.read_csv(PROJECT_ROOT / 'metadata' / 'manifest.csv', keep_default_na=False)
df = manifest[['image', 'level', 'split']].merge(scores, on='image', validate='one_to_one')
df['rejected'] = df['p_reject'] >= THRESHOLD

graded = df[df['eyeq_quality'] != '']
ungraded = df[df['eyeq_quality'] == '']
print(f'z oceną EyeQ: {len(graded)}, bez oceny: {len(ungraded)}')
print(f'odrzucone przy progu {THRESHOLD}: z oceną {graded.rejected.mean():.1%}, bez oceny {ungraded.rejected.mean():.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
bins = np.linspace(0, 1, 51)
for quality in ('Good', 'Usable', 'Reject'):
    axes[0].hist(graded.loc[graded.eyeq_quality == quality, 'p_reject'], bins=bins, alpha=0.6, label=quality)
axes[0].set_yscale('log')
axes[0].axvline(THRESHOLD, color='k', ls='--')
axes[0].set_title('p_reject dla zdjęć z oceną EyeQ (walidacja krzyżowa)')
axes[0].legend()
axes[1].hist(ungraded['p_reject'], bins=bins)
axes[1].set_yscale('log')
axes[1].axvline(THRESHOLD, color='k', ls='--')
axes[1].set_title('p_reject dla zdjęć bez oceny EyeQ')
plt.tight_layout()
plt.show()

table = pd.crosstab(ungraded['level'], ungraded['rejected'], margins=True)
table.columns = ['zostaje', 'odrzucone', 'razem']
table['% odrzuconych'] = (table['odrzucone'] / table['razem'] * 100).round(1)
table

In [ ]:
def load_thumb(image_name: str, size: int = 256) -> np.ndarray:
    image = cv2.imread(str(IMAGES_DIR / f'{image_name}.jpeg'))
    if image is None:
        return np.zeros((size, size, 3), dtype=np.uint8)
    return cv2.resize(cv2.cvtColor(image, cv2.COLOR_BGR2RGB), (size, size), interpolation=cv2.INTER_AREA)


def show_grid(rows: pd.DataFrame, title: str, cols: int = 8) -> None:
    if len(rows) == 0:
        print(f'{title}: brak zdjęć')
        return
    n_rows = int(np.ceil(len(rows) / cols))
    fig, axes = plt.subplots(n_rows, cols, figsize=(cols * 2.3, n_rows * 2.55))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis('off')
    for ax, row in zip(axes, rows.itertuples()):
        ax.imshow(load_thumb(row.image))
        ax.set_title(f'{row.image}\nklasa {row.level} | p={row.p_reject:.2f}', fontsize=8)
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()


def sample(rows: pd.DataFrame, n: int) -> pd.DataFrame:
    return rows.sample(min(n, len(rows)), random_state=SEED).sort_values('p_reject')

## 1. Odrzucone (bez oceny EyeQ) — losowe 32

In [ ]:
show_grid(sample(ungraded[ungraded.rejected], 32), f'odrzucone, p_reject ≥ {THRESHOLD}: losowe 32 z {ungraded.rejected.sum()}')

## 2. Zachowane (bez oceny EyeQ) — losowe 32

In [ ]:
show_grid(sample(ungraded[~ungraded.rejected], 32), f'zachowane, p_reject < {THRESHOLD}: losowe 32 z {(~ungraded.rejected).sum()}')

## 3. Przy progu — najtrudniejsze decyzje

Górne wiersze: tuż **poniżej** progu (zostają), dolne: tuż **powyżej** (odpadają).
Jeśli zdjęcia powyżej progu wyglądają na w porządku, próg warto podnieść.

In [ ]:
below = ungraded[(ungraded.p_reject >= THRESHOLD - 0.1) & (ungraded.p_reject < THRESHOLD)]
above = ungraded[(ungraded.p_reject >= THRESHOLD) & (ungraded.p_reject < THRESHOLD + 0.1)]
show_grid(sample(below, 16), f'zostają: p_reject w [{THRESHOLD - 0.1:.1f}, {THRESHOLD:.1f})  ({len(below)} zdjęć)')
show_grid(sample(above, 16), f'odpadają: p_reject w [{THRESHOLD:.1f}, {THRESHOLD + 0.1:.1f})  ({len(above)} zdjęć)')

## 4. Odrzucone w klasach 3–4

Tu najbardziej boli strata danych. Czy zmiany chorobowe są mimo słabej jakości dobrze widoczne?

In [ ]:
show_grid(sample(ungraded[ungraded.rejected & (ungraded.level == 3)], 16), 'odrzucone, klasa 3')
show_grid(sample(ungraded[ungraded.rejected & (ungraded.level == 4)], 16), 'odrzucone, klasa 4')

## 5. Gdzie model nie zgadza się z EyeQ (zdjęcia z oceną, walidacja krzyżowa)

Pozwala ocenić, jakie błędy model robi.

In [ ]:
fp = graded[graded.rejected & (graded.eyeq_quality != 'Reject')]
fn = graded[~graded.rejected & (graded.eyeq_quality == 'Reject')]
show_grid(sample(fp, 16), f'model odrzuca, EyeQ: Good/Usable ({len(fp)} zdjęć)')
show_grid(sample(fn, 16), f'model zostawia, EyeQ: Reject ({len(fn)} zdjęć)')